# Clase 2 – Ingeniería y Selección de Atributos
### Maestría en Inteligencia Artificial y Análisis de Datos – FP‑UNA

En esta clase profundizaremos en dos pilares fundamentales del aprendizaje automático:

1. **Ingeniería de atributos**: transformar, limpiar y preparar los datos.
2. **Selección de atributos**: elegir las variables más relevantes para mejorar el rendimiento del modelo.

Ambos procesos son esenciales para evitar sobreajuste, reducir dimensionalidad y mejorar la interpretabilidad.

## Formalización general
Dado un conjunto de datos:

$$
\mathcal{D} = \{(x_i, y_i)\}_{i=1}^n,
$$

donde cada vector de atributos es:

$$
x_i = (x_{i1}, x_{i2}, \dots, x_{id}) \in \mathbb{R}^d,
$$

nuestro objetivo es encontrar una transformación:

$$
T: \mathbb{R}^d \rightarrow \mathbb{R}^k,
$$

tal que los atributos transformados **maximicen la información útil** para el modelo.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold, SelectKBest, SelectPercentile
from sklearn.feature_selection import chi2, f_classif, mutual_info_classif
from sklearn.feature_selection import RFE
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.datasets import make_classification

# 1. Detección y eliminación de outliers

Los **outliers** son valores que se alejan significativamente del comportamiento general de los datos.

Una forma común de detectarlos es mediante el **Z-score**:

$$
z_i = \frac{x_i - \mu}{\sigma}
$$

Si $|z_i| > 3$, el punto suele considerarse un outlier.

In [ ]:
np.random.seed(42)
X = np.random.rand(100, 2) * 10
y = np.random.randint(0, 2, 100)

X[0, 0] = 100
X[1, 1] = -50

df = pd.DataFrame(np.column_stack((X, y)), columns=['feature1', 'feature2', 'target'])
df.head()

### Eliminación de outliers usando Z-score

In [ ]:
def remove_outliers(df, column, threshold=3):
    z = np.abs((df[column] - df[column].mean()) / df[column].std())
    return df[z < threshold]

df_clean = remove_outliers(df, 'feature1')
df_clean = remove_outliers(df_clean, 'feature2')
df_clean.head()

# 2. Imputación de valores faltantes

Cuando existen valores faltantes, una estrategia común es reemplazarlos por la **media**:

$$
\hat{x}_j = \frac{1}{n} \sum_{i=1}^n x_{ij}
$$

Esto preserva la escala general del atributo.

In [ ]:
df_missing = df_clean.copy()
df_missing.loc[5:7, 'feature1'] = None

imputer = SimpleImputer(strategy='mean')
df_imputed = df_missing.copy()
df_imputed[['feature1','feature2']] = imputer.fit_transform(df_missing[['feature1','feature2']])
df_imputed.head()

# 3. Normalización y estandarización

La estandarización transforma cada atributo según:

$$
x' = \frac{x - \mu}{\sigma}
$$

Esto es fundamental para modelos sensibles a la escala.

In [ ]:
scaler = StandardScaler()
scaled = scaler.fit_transform(df_imputed[['feature1','feature2']])
pd.DataFrame(scaled, columns=['feature1','feature2']).head()

# 4. Codificación de variables categóricas

El **One-Hot Encoding** transforma una variable categórica en un vector binario.

Formalmente, si una variable toma valores en un conjunto finito $C = \{c_1, \dots, c_k\}$, entonces:

$$
\text{onehot}(c_j) = e_j \in \mathbb{R}^k
$$

In [ ]:
data = [['A1','B1','C1'], ['A2','B2','C2'], ['A3','B2','C3']]
df_cat = pd.DataFrame(data, columns=['FA','FB','FC'])

encoder = OneHotEncoder()
encoded = encoder.fit_transform(df_cat)
pd.DataFrame.sparse.from_spmatrix(encoded).head()

# 5. Reducción de dimensionalidad con PCA

PCA busca una transformación lineal:

$$
Z = XW
$$

donde las columnas de $W$ son los autovectores de la matriz de covarianza.


In [ ]:
pca = PCA(n_components=2)
components = pca.fit_transform(scaled)
pd.DataFrame(components, columns=['PC1','PC2']).head()

# 6. Selección de atributos

La selección de atributos busca identificar las variables más informativas.

Ejemplo de medida supervisada:

$$
F = \frac{\text{var}(\bar{x}_1 - \bar{x}_2)}{\text{var}(x)}
$$

# 6. Selección de atributos

La **selección de atributos** es el proceso de identificar el subconjunto más relevante de variables para un modelo predictivo.  
Su objetivo es mejorar el rendimiento, reducir el sobreajuste y aumentar la interpretabilidad.

Formalmente, dado un conjunto de atributos:

$X = [x_1, x_2, \dots, x_d]$

la selección de atributos busca un subconjunto óptimo:

$X^* \subseteq X$

tal que se maximice una función de calidad del modelo:

$X^* = \arg\max_{S \subseteq X} \; \mathcal{Q}(f_S)$

donde $f_S$ es el modelo entrenado solo con los atributos en $S$.

Existen tres grandes familias de métodos:

1. **Métodos no supervisados**  
2. **Métodos supervisados**  
3. **Métodos wrapper y embedded**

A continuación exploramos cada uno con ejemplos prácticos.


## 6.1 Métodos no supervisados: Variance Threshold

Los atributos con **varianza muy baja** aportan poca información, ya que casi no cambian entre observaciones.

La varianza de un atributo $x$ se define como:

$\mathrm{Var}(x) = \frac{1}{n} \sum_{i=1}^n (x_i - \bar{x})^2$

Si $\mathrm{Var}(x) \approx 0$, entonces el atributo es casi constante y puede eliminarse sin pérdida de información.

El método `VarianceThreshold` elimina automáticamente atributos cuya varianza esté por debajo de un umbral.


In [ ]:
X_sel, y_sel = make_classification(n_samples=300, n_features=10, n_redundant=3)

sel = VarianceThreshold(threshold=0.1)
X_var = sel.fit_transform(X_sel)
X_var.shape

## 6.2 Métodos supervisados: ANOVA F-score

El **ANOVA F-score** mide la relación entre cada atributo y la variable objetivo en problemas de clasificación.

Para un atributo $x$, el estadístico F se define como:

$F = \frac{\text{Var}(\text{media entre clases})}{\text{Var}(\text{dentro de clases})}$

Un valor alto de $F$ indica que el atributo separa bien las clases.

Este método evalúa cada atributo de forma **univariada**, sin considerar interacciones.


In [ ]:
f_scores = f_classif(X_sel, y_sel)[0]
f_scores

## 6.3 Información Mutua (Mutual Information)

La **información mutua** mide cuánta información comparte un atributo con la variable objetivo.

Formalmente:

$I(X; Y) = \sum_{x} \sum_{y} p(x, y) \log \frac{p(x, y)}{p(x)p(y)}$

Si $I(X;Y) = 0$, entonces el atributo no aporta información sobre la clase.  
Si es alto, el atributo es relevante.

A diferencia de ANOVA, **captura relaciones no lineales**.


In [ ]:
mi_scores = mutual_info_classif(X_sel, y_sel)
mi_scores

## 6.4 Métodos Wrapper: RFE (Recursive Feature Elimination)

Los métodos wrapper utilizan un modelo para evaluar subconjuntos de atributos.

RFE funciona así:

1. Entrena un modelo con todos los atributos.  
2. Calcula la importancia de cada atributo.  
3. Elimina el menos importante.  
4. Repite hasta obtener el número deseado de atributos.

Formalmente, si $w_j$ es la importancia del atributo $x_j$, RFE elimina:

$x_{j^*} = \arg\min_j |w_j|$

Este método es más costoso computacionalmente, pero suele producir mejores resultados.


In [ ]:
svc = SVC(kernel="linear")
rfe = RFE(svc, n_features_to_select=4)
X_rfe = rfe.fit_transform(X_sel, y_sel)
X_rfe.shape

## 6.5 Métodos Embedded

Los métodos embedded integran la selección de atributos dentro del propio algoritmo de aprendizaje.

### Árboles de decisión y bosques aleatorios

Los modelos basados en árboles calculan la importancia de cada atributo según la reducción de impureza:

$\Delta i = i(\text{padre}) - \sum_k \frac{N_k}{N} i(\text{hijo}_k)$

donde $i(\cdot)$ es una medida como Gini o entropía.

### SVM lineal

Un SVM lineal aprende un vector de pesos $w$.  
Los atributos con pesos grandes son más relevantes:

$\text{importancia}(x_j) = |w_j|$

Estos métodos son eficientes y funcionan bien en alta dimensionalidad.


In [ ]:
clf = ExtraTreesClassifier(n_estimators=50)
clf.fit(X_sel, y_sel)
clf.feature_importances_

# 🎮 Desafíos Gamificados

## 🔍 Desafío 1 — Cazador de Outliers
Elimina outliers y compara el rendimiento de un modelo antes y después.

## 🧠 Desafío 2 — Elige tus mejores atributos
Usa ANOVA, Chi2 o MI para seleccionar atributos.

## 🎯 Desafío 3 — Reduce y gana
Aplica PCA y visualiza los datos en 2D.

# 📊 Rúbrica de Evaluación – Clase 2

| Criterio | Excelente (5) | Bueno (4) | Aceptable (3) | Insuficiente (1–2) |
|---------|---------------|-----------|---------------|---------------------|
| Ingeniería de atributos | Aplica todas las técnicas correctamente | Aplica varias | Aplica parcialmente | Incorrecto |
| Selección de atributos | Usa 3+ métodos | Usa 2 métodos | Usa 1 método | No aplica |
| Interpretación | Clara y profunda | Correcta | Parcial | Incorrecta |
| Código | Limpio y funcional | Funciona | Funciona con errores | No funciona |
| Creatividad | Soluciones originales | Correcto | Básico | No entrega |

**Total: 25 puntos**